In [1]:
from data import *
import numpy as np
import pandas as pd

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/04 13:04:11 WARN Utils: Your hostname, Siddharths-MacBook-Air-3.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.137 instead (on interface en0)
26/01/04 13:04:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/04 13:04:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
sales_pred = pd.read_csv('./data/m5-forecasting-accuracy/test.csv')

In [5]:
# sales_test.insert(0, 'id', sales_test['item_id'].str.cat(sales_test['store_id'], sep='_'))
sales_test = sales_test.drop(columns=['item_id', 'store_id', 'dept_id', 'cat_id', 'state_id'])
sales_test

,id,d_1942,d_1943,d_1944,d_1945,d_1946,d_1947,d_1948,d_1949,d_1950,...,d_1960,d_1961,d_1962,d_1963,d_1964,d_1965,d_1966,d_1967,d_1968,d_1969
0,HOBBIES_1_001_CA_1,2,0,1,0,0,1,4,3,0,...,2,1,2,0,0,1,0,1,3,1
1,HOBBIES_1_002_CA_1,0,2,0,1,0,1,0,1,0,...,1,0,0,1,0,0,2,1,1,0
2,HOBBIES_1_003_CA_1,0,0,0,0,0,1,0,0,0,...,1,3,2,1,0,2,1,0,1,1
3,HOBBIES_1_004_CA_1,0,0,1,0,6,3,3,2,1,...,3,3,4,2,1,6,3,1,4,3
4,HOBBIES_1_005_CA_1,2,0,1,1,2,4,0,2,2,...,0,1,2,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3,0,0,1,1,0,2,0,0,0,...,1,4,0,0,1,1,1,1,1,0
30486,FOODS_3_824_WI_3,0,0,1,1,0,1,0,0,0,...,4,1,0,1,0,1,0,0,0,0
30487,FOODS_3_825_WI_3,1,0,0,0,0,0,0,1,1,...,1,1,2,1,2,1,1,1,1,0
30488,FOODS_3_826_WI_3,0,0,0,0,1,1,1,2,1,...,0,0,2,0,1,3,0,2,1,5


In [14]:
sales_id = sales_pred.iloc[:, :-28]
sqrd_diff_values = (sales_pred.iloc[:, -28:].values - sales_test.iloc[:, -28:].values) ** 2
sales_diff = pd.DataFrame(
    sqrd_diff_values, 
    index=sales_pred.index, 
    columns=sales_pred.columns[-28:]
)
sales_diff = pd.concat([sales_id, sales_diff], axis=1)
sales_diff['total'] = sales_diff.drop(columns=['id']).sum(axis=1)
sales_diff

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F20,F21,F22,F23,F24,F25,F26,F27,F28,total
0,HOBBIES_1_001_CA_1,1.340943,0.606564,0.049284,0.500348,0.913858,0.067274,8.625929,4.033164,0.648866,...,0.078357,0.811067,0.775686,0.660232,0.056907,0.678745,0.001235,3.025330,0.066463,35.185455
1,HOBBIES_1_002_CA_1,0.048387,3.230425,0.043216,0.654958,0.049410,0.538852,0.070003,0.578081,0.032608,...,0.103440,0.101615,0.596946,0.054342,0.058529,3.082432,0.507170,0.448646,0.069767,13.086100
2,HOBBIES_1_003_CA_1,0.237785,0.188234,0.274471,0.273612,0.455719,0.012235,0.598711,0.549726,0.242856,...,5.117892,1.576658,0.309192,0.153900,2.350318,0.290242,0.397597,0.064657,0.140970,15.547385
3,HOBBIES_1_004_CA_1,2.085627,1.762711,0.144378,1.834556,18.231942,0.377539,0.333689,0.007163,0.102712,...,0.454973,1.752000,0.120295,0.073009,21.446929,2.823741,0.539384,1.926906,0.002425,114.974417
4,HOBBIES_1_005_CA_1,1.088472,0.846719,0.021853,0.022668,1.045345,6.555883,1.925119,1.309357,1.226737,...,0.265176,0.203211,0.000024,1.027910,1.095676,1.150179,1.530802,2.793519,1.309420,33.430525
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3,0.170972,0.165397,0.367099,0.320280,0.245493,2.019665,0.434044,0.172971,0.192533,...,10.598239,0.682754,0.358098,0.168491,0.177334,0.255748,0.224113,0.131356,0.555481,20.751227
30486,FOODS_3_824_WI_3,0.063496,0.064530,0.566121,0.584006,0.047487,0.553749,0.075553,0.048810,0.066110,...,0.380027,0.153090,0.468231,0.121860,0.404034,0.078637,0.061951,0.095418,0.121696,19.814089
30487,FOODS_3_825_WI_3,0.151712,0.274020,0.239768,0.228852,0.294811,0.468975,0.537070,0.212441,0.262753,...,0.033334,0.447749,0.001946,0.862691,0.012158,0.065170,0.074179,0.033796,0.879927,13.742139
30488,FOODS_3_826_WI_3,1.045705,1.167385,0.940571,0.859261,0.000737,0.043224,0.063272,0.680755,0.009446,...,1.872686,0.286312,1.505077,0.359926,2.329864,1.588788,0.539112,0.212248,13.075717,64.035768
